In [103]:
import subprocess, os, json, requests, re
from evalplus.data import get_human_eval_plus

In [104]:
path = "../data/processed/selfdebug/qwen_3b"
humaneval_filename = "humaneval_solutions.jsonl"
ollama_url = "http://host.docker.internal:11434/api/generate"
max_iterations = 3

In [105]:
def test_solution(solution_code, test_code, entry_point, timeout=5):
    script = f"{solution_code}\n\n{test_code}"
    if not "def check(" in test_code:
        script = f"{script}\n\ncheck({entry_point})\n\n"

    try:
        result = subprocess.run(
            ["python3", "-c", script],
            capture_output=True,
            text=True,
            timeout=timeout
        )

        if result.returncode == 0:
            return True, ""
        else:
            return False, result.stderr.strip()

    except subprocess.TimeoutExpired:
        return False, "TimeoutExpired: the code entered an infinite loop, or spent too much time."
    except Exception as e:
        return False, f"Execution Error: {str(e)}"

    

In [106]:
def clean_code(code):
    regex = r"```python\n(.*?)\n```"
    match = re.search(regex, code, re.DOTALL)
    return match.group(1) if match else code

In [ ]:
def generate_solution(problems, output_file):
    completed_tasks = set()
    if not os.path.exists(path):
        os.makedirs(path)

    if os.path.exists(output_file):
        with open(output_file, 'r') as f:
            for line in f:
                completed_tasks.add(json.loads(line.strip())["task_id"])

    print(f"loaded {len(completed_tasks)} solutions.\n")
    if(len(problems) - len(completed_tasks) == 0):
        print("solutions already generated.\n")
        return

    with open(output_file, 'a') as f:
        for task_id in problems.keys():
            problem = problems[task_id]
            prompt = problem["prompt"]

            i = 0
            while True:
                payload = {
                    "model": "qwen2.5-coder:3b",
                    "prompt": prompt,
                    "stream": False
                }
                response = requests.post(ollama_url, json=payload).json()
                generated_code = clean_code(response.get("response", ""))

                test_result, test_error = test_solution(generated_code, problem["test"], problem["entry_point"])
                if test_result or i > max_iterations:
                    break

                i += 1
                print(f"problem {task_id} has failed the test")
                prompt = f"The following code is not correct: {generated_code}\n\nDuring execution the following exception has been encountered: {test_error}\n\nRewrite the function fixing the error."

            f.write(json.dumps({
                "task_id": task_id,
                "solution": generated_code
            }) + "\n")


In [108]:
humaneval_problems = get_human_eval_plus()
generate_solution(humaneval_problems, f"{path}/{humaneval_filename}")

loaded 0 solutions.

problem HumanEval/3 has failed the test
problem HumanEval/4 has failed the test
problem HumanEval/4 has failed the test
problem HumanEval/4 has failed the test
problem HumanEval/4 has failed the test
problem HumanEval/64 has failed the test
problem HumanEval/64 has failed the test
problem HumanEval/64 has failed the test
problem HumanEval/64 has failed the test
problem HumanEval/132 has failed the test
problem HumanEval/132 has failed the test
problem HumanEval/132 has failed the test


In [109]:
!evalplus.evaluate --dataset humaneval --samples ../data/processed/selfdebug/qwen_3b/humaneval_solutions.jsonl

Load from ground-truth from /home/vscode/.cache/evalplus/fe585eb4df8c88d844eeb463ea4d0302.pkl
Reading samples...
164it [00:00, 1495.36it/s]
100%|████████████████████████████████████████| 164/164 [00:08<00:00, 20.43it/s]
humaneval (base tests)
pass@1:	0.811
humaneval+ (base + extra tests)
pass@1:	0.762
